# Import

In [6]:

import requests
from bs4 import BeautifulSoup

In [1]:
headers = {
    'User-Agent': 'MonBotApprentissage/1.0 (formation scraping; contact@example.com)',
    'Accept-Language': 'fr-FR,fr;q=0.9',
    'Accept': 'text/html,application/xhtml+xml',
    'From': 'contact@example.com'  # Email de contact (bonne pratique)
}


url = "https://fr.wikipedia.org/wiki/Classement_mondial_des_entreprises_leader_par_secteur"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
print(soup)

NameError: name 'requests' is not defined

# Étape 1 : Cibler la table (Le Conteneur)
Avant de chercher les lignes, on isole la table pour ne pas récupérer par erreur des lignes d'un autre tableau sur la même page.

In [8]:
# On cherche la table qui a la classe "wikitable"
table = soup.select_one("table.wikitable:nth-of-type(2)")
# nth-of-type(2) permet de sélectionner la 2ème table avec cette classe

# Étape 2 : Trouver la première ligne (L'en-tête)
La première ligne contient généralement les titres des colonnes (balises <th>). On utilise select_one car on n'en veut qu'une.

In [9]:
# 1. On cherche le premier "tr" (table row) à l'intérieur de notre table
premiere_ligne = table.select_one("tr")
print(premiere_ligne)

<tr>
<th scope="col">Rang</th>
<th scope="col">Entreprise</th>
<th scope="col">Pays</th>
<th scope="col">Capitalisation boursière <br/>(en milliards de <a href="/wiki/Dollar_am%C3%A9ricain" title="Dollar américain">$</a>)
</th></tr>


In [10]:
# 2. On crée une liste vide pour stocker les noms des colonnes
noms_des_colonnes = []

# 3. On récupère toutes les balises <th> (Table Headers) dans cette ligne
tous_les_th = premiere_ligne.select("th")

# 4. On parcourt chaque balise trouvée
for th in tous_les_th:
    # On récupère le texte brut et On nettoie le texte (enlève les espaces et les sauts de ligne \n)
    texte_brut = th.text.strip()
    
    # On ajoute ce texte propre à notre liste
    noms_des_colonnes.append(texte_brut)

# 5. On affiche le résultat final
print(f"Colonnes trouvées : {noms_des_colonnes}")

Colonnes trouvées : ['Rang', 'Entreprise', 'Pays', 'Capitalisation boursière (en milliards de $)']


# Étape 3 : Créer une boucle pour les autres lignes
Pour récupérer toutes les lignes de données sans la première (l'en-tête), on utilise select qui renvoie une liste, puis on utilise le "slicing" Python [1:] pour ignorer le premier élément.

In [11]:
# 1. On récupère TOUTES les lignes (tr)
toutes_les_lignes = table.select("tr")

# Visualiser le "Bloc" de la ligne
# Avant d'aller chercher le texte dans les cases (td), on affiche l'objet ligne lui-même.

for ligne in toutes_les_lignes[1:]:
    # On affiche l'objet ligne brut pour voir ce qu'il contient
    print("--- DÉBUT DE LIGNE ---")
    print(ligne) 
    print("--- FIN DE LIGNE ---\n")

--- DÉBUT DE LIGNE ---
<tr>
<td>1</td>
<td><a href="/wiki/Nestl%C3%A9" title="Nestlé">Nestlé</a></td>
<td><span class="datasortkey" data-sort-value="Suisse"><span class="flagicon nowrap"><span class="mw-image-border noviewer" typeof="mw:File"><a class="mw-file-description" href="/wiki/Fichier:Flag_of_Switzerland.svg" title="Drapeau de la Suisse"><img alt="Drapeau de la Suisse" class="mw-file-element" data-file-height="512" data-file-width="512" decoding="async" height="15" src="//upload.wikimedia.org/wikipedia/commons/thumb/f/f3/Flag_of_Switzerland.svg/20px-Flag_of_Switzerland.svg.png" srcset="//upload.wikimedia.org/wikipedia/commons/thumb/f/f3/Flag_of_Switzerland.svg/40px-Flag_of_Switzerland.svg.png 1.5x" width="15"/></a></span> </span><a href="/wiki/Suisse" title="Suisse">Suisse</a></span></td>
<td>372
</td></tr>
--- FIN DE LIGNE ---

--- DÉBUT DE LIGNE ---
<tr>
<td>2</td>
<td><a href="/wiki/PepsiCo" title="PepsiCo">PepsiCo</a></td>
<td><span class="nowrap"><span class="datasortkey" 

In [13]:
# 2. On boucle à partir de la deuxième (index 1) jusqu'à la fin

#initialisation d'une liste pour stocker les données extraites
donnees = []

for ligne in toutes_les_lignes[1:]:
    # À l'intérieur de chaque ligne, on cherche les cellules (td)
    cellules = ligne.select("td")
    
    if cellules: # Sécurité pour vérifier que la ligne n'est pas vide
        rang = cellules[0].text.strip()
        entreprise = cellules[1].text.strip()
        pays = cellules[2].text.strip()
        ca = cellules[3].text.strip()

        # On ajoute les données extraites à notre liste
        donnees.append((rang, entreprise, pays, ca))
        print(f"{rang} | {entreprise} ({pays}) -> CA: {ca} Mrd€")

1 | Nestlé (Suisse) -> CA: 372 Mrd€
2 | PepsiCo (États-Unis) -> CA: 240 Mrd€
3 | McDonald's (États-Unis) -> CA: 199 Mrd€
4 | Unilever (Royaume-Uni /  Pays-Bas) -> CA: 137 Mrd€
5 | Mondelez (États-Unis) -> CA: 94 Mrd€
6 | Cargill (États-Unis) -> CA:  Mrd€
7 | Hindustan Unilever (Inde) -> CA: 76 Mrd€
8 | Foshan Haitian Flavouring & Food Co (en) (Chine) -> CA: 72 Mrd€
9 | Yihai Kerry Arawana (Chine) -> CA: 55 Mrd€


In [ ]:
import pandas as pd

# Créer un DataFrame pandas pour mieux visualiser les données
df_wiki = pd.DataFrame(donnees, columns = noms_des_colonnes)
print(df_wiki.head())

   0           1                        2    3
0  1      Nestlé                   Suisse  372
1  2     PepsiCo               États-Unis  240
2  3  McDonald's               États-Unis  199
3  4    Unilever  Royaume-Uni /  Pays-Bas  137
4  5    Mondelez               États-Unis   94


In [ ]:
# On cherche tous les 2èmes <td> de chaque ligne directement
nom_entreprise = table.select("tr td span.datasortkey[data-sort-value]")

for nom in nom_entreprise:
    print(nom.text.strip())

Suisse
États-Unis
États-Unis
Royaume-Uni
Pays-Bas
États-Unis
États-Unis
Inde
Chine
Chine
